# Genre and Studio Performance Optimization

**Objective:** Translate the publisher-optimization framework (Kellogg/Hult SEM analytics: Cost-per-Click,
Probability of Booking, Net Revenue, ROA, and the four-quadrant funding strategy) onto real movie
industry financial data, treating each genre and each production studio the way that framework treats
an advertising publisher.

**Metric translation:**
- Cost per Click -> Average Budget per Film
- Probability of Booking -> Probability of Profitability (share of films that earned back their budget)
- Net Revenue -> Total Revenue - Total Budget
- Return on Ad Spend (ROA) -> Net Revenue / Total Budget
- Publisher funding quadrant (Probability of Booking vs. CPC, bubble = funding) -> Genre/Studio funding
  quadrant (Probability of Profitability vs. Average Budget, bubble = total dollars invested)

**Output:** `genre_stats.csv`, `studio_stats.csv`, and a funding-strategy classification for each.

## 1. Setup

In [1]:
import pandas as pd
import numpy as np
import ast
import matplotlib.pyplot as plt

pd.set_option('display.width', 140)
plt.style.use('seaborn-v0_8-whitegrid')

tmdb = pd.read_csv("../data/tmdb_clean.csv")
tmdb['genre_list'] = tmdb['genre_list'].apply(ast.literal_eval)
tmdb['studio_list'] = tmdb['studio_list'].apply(ast.literal_eval)
print("Loaded", len(tmdb), "financially-tracked films.")

Loaded 3213 financially-tracked films.


## 2. Genre-Level Metrics

In [2]:
genre_rows = tmdb.explode('genre_list').rename(columns={'genre_list': 'genre'}).dropna(subset=['genre'])

genre_stats = genre_rows.groupby('genre').agg(
    n_movies=('id', 'count'),
    total_budget=('budget', 'sum'),
    total_revenue=('revenue', 'sum'),
    avg_budget=('budget', 'mean'),
    avg_revenue=('revenue', 'mean'),
    prob_profitable=('profitable', 'mean'),
).reset_index()
genre_stats['net_revenue'] = genre_stats['total_revenue'] - genre_stats['total_budget']
genre_stats['roa'] = genre_stats['net_revenue'] / genre_stats['total_budget']
genre_stats = genre_stats[genre_stats['n_movies'] >= 20].sort_values('roa', ascending=False).reset_index(drop=True)
genre_stats.round(3)

,genre,n_movies,total_budget,total_revenue,avg_budget,avg_revenue,prob_profitable,net_revenue,roa
0,Documentary,37,206070045,980422111,5.569461e+06,2.649789e+07,0.703,774352066,3.758
1,Animation,186,15195420776,52436463168,8.169581e+07,2.819165e+08,0.806,37241042392,2.451
2,Family,364,24908238176,82427900949,6.842923e+07,2.264503e+08,0.816,57519662773,2.309
3,Music,111,2660647585,8776082899,2.396980e+07,7.906381e+07,0.757,6115435314,2.298
4,Horror,328,6825123183,22468437558,2.080830e+07,6.850133e+07,0.820,15643314375,2.292
5,Romance,570,16243424806,53130388208,2.849724e+07,9.321121e+07,0.761,36886963402,2.271
6,Adventure,660,50825869013,163667924225,7.700889e+07,2.479817e+08,0.794,112842055212,2.220
7,Fantasy,340,25869185826,81497220347,7.608584e+07,2.396977e+08,0.803,55628034521,2.150
8,Science Fiction,428,27072838057,81352803603,6.325429e+07,1.900766e+08,0.766,54279965546,2.005
9,Comedy,1104,40132554630,120403594594,3.635195e+07,1.090612e+08,0.764,80271039964,2.000


## 3. Genre Funding Strategy Quadrant

Each genre is placed into one of four quadrants relative to the cross-genre average probability of
profitability and average budget, then assigned the corresponding strategy from the source framework.

In [3]:
x_avg = genre_stats['prob_profitable'].mean()
y_avg = genre_stats['avg_budget'].mean()

def quadrant_strategy(prob, budget, x_ref, y_ref):
    high_prob = prob >= x_ref
    high_cost = budget >= y_ref
    if high_prob and high_cost:
        return "Adjust Tactics"
    elif high_prob and not high_cost:
        return "Fund More"
    elif not high_prob and high_cost:
        return "Fund Less / Cut"
    else:
        return "Improve Positioning"

genre_stats['strategy'] = genre_stats.apply(
    lambda r: quadrant_strategy(r['prob_profitable'], r['avg_budget'], x_avg, y_avg), axis=1
)
print(f"Reference lines: avg probability of profitability = {x_avg:.3f}, avg budget = ${y_avg/1e6:.1f}M")
genre_stats[['genre','n_movies','prob_profitable','avg_budget','roa','strategy']].round(3)

Reference lines: avg probability of profitability = 0.755, avg budget = $44.7M


,genre,n_movies,prob_profitable,avg_budget,roa,strategy
0,Documentary,37,0.703,5.569461e+06,3.758,Improve Positioning
1,Animation,186,0.806,8.169581e+07,2.451,Adjust Tactics
2,Family,364,0.816,6.842923e+07,2.309,Adjust Tactics
3,Music,111,0.757,2.396980e+07,2.298,Fund More
4,Horror,328,0.820,2.080830e+07,2.292,Fund More
5,Romance,570,0.761,2.849724e+07,2.271,Fund More
6,Adventure,660,0.794,7.700889e+07,2.220,Adjust Tactics
7,Fantasy,340,0.803,7.608584e+07,2.150,Adjust Tactics
8,Science Fiction,428,0.766,6.325429e+07,2.005,Adjust Tactics
9,Comedy,1104,0.764,3.635195e+07,2.000,Fund More


In [4]:
fig, ax = plt.subplots(figsize=(9, 6.5))
colors = {'Fund More': '#4C9A2A', 'Adjust Tactics': '#E8A33D', 'Fund Less / Cut': '#C1443C', 'Improve Positioning': '#5B7DB1'}
for _, row in genre_stats.iterrows():
    ax.scatter(row['prob_profitable'], row['avg_budget']/1e6, s=row['total_budget']/2e6,
               color=colors[row['strategy']], alpha=0.65, edgecolor='black', linewidth=0.5)
    ax.annotate(row['genre'], (row['prob_profitable'], row['avg_budget']/1e6), fontsize=8,
                xytext=(4, 4), textcoords='offset points')
ax.axvline(x_avg, color='gray', linestyle='--', linewidth=1)
ax.axhline(y_avg/1e6, color='gray', linestyle='--', linewidth=1)
ax.set_xlabel('Probability of Profitability')
ax.set_ylabel('Average Budget ($M)')
ax.set_title('Genre Funding Strategy Quadrant (bubble size = total historical investment)')
handles = [plt.Line2D([0],[0], marker='o', color='w', markerfacecolor=c, markersize=10, label=k) for k, c in colors.items()]
ax.legend(handles=handles, loc='upper left', fontsize=8)
plt.tight_layout()
plt.savefig('genre_quadrant.png', dpi=100)
plt.show()

<Figure size 900x650 with 1 Axes>

## 4. Studio-Level Metrics

In [5]:
studio_rows = tmdb.explode('studio_list').rename(columns={'studio_list': 'studio'}).dropna(subset=['studio'])

studio_stats = studio_rows.groupby('studio').agg(
    n_movies=('id', 'count'),
    total_budget=('budget', 'sum'),
    total_revenue=('revenue', 'sum'),
    avg_budget=('budget', 'mean'),
    avg_revenue=('revenue', 'mean'),
    prob_profitable=('profitable', 'mean'),
).reset_index()
studio_stats['net_revenue'] = studio_stats['total_revenue'] - studio_stats['total_budget']
studio_stats['roa'] = studio_stats['net_revenue'] / studio_stats['total_budget']
studio_stats = studio_stats[studio_stats['n_movies'] >= 10].sort_values('roa', ascending=False).reset_index(drop=True)

x_avg_s = studio_stats['prob_profitable'].mean()
y_avg_s = studio_stats['avg_budget'].mean()
studio_stats['strategy'] = studio_stats.apply(
    lambda r: quadrant_strategy(r['prob_profitable'], r['avg_budget'], x_avg_s, y_avg_s), axis=1
)

print("Studios with 10+ films in the dataset:", len(studio_stats))
studio_stats.head(15).round(3)

Studios with 10+ films in the dataset: 147


,studio,n_movies,total_budget,total_revenue,avg_budget,avg_revenue,prob_profitable,net_revenue,roa,strategy
0,Blumhouse Productions,20,86215000,1794436022,4.310750e+06,8.972180e+07,0.950,1708221022,19.814,Fund More
1,Lucasfilm,14,796427000,6606736700,5.688764e+07,4.719098e+08,0.929,5810309700,7.295,Adjust Tactics
2,Twentieth Century Fox Animation,11,891000000,4866552091,8.100000e+07,4.424138e+08,1.000,3975552091,4.462,Adjust Tactics
3,Sunswept Entertainment,10,644000000,3343422001,6.440000e+07,3.343422e+08,0.900,2699422001,4.192,Adjust Tactics
4,Fox Searchlight Pictures,47,520000159,2651120954,1.106383e+07,5.640683e+07,0.851,2131120795,4.098,Fund More
5,Dentsu,12,1062000000,5337121421,8.850000e+07,4.447601e+08,0.833,4275121421,4.026,Adjust Tactics
6,Orion Pictures,10,186400000,923755895,1.864000e+07,9.237559e+07,0.700,737355895,3.956,Improve Positioning
7,WingNut Films,11,1453000000,7081402502,1.320909e+08,6.437639e+08,0.909,5628402502,3.874,Adjust Tactics
8,Platinum Dunes,12,844000000,3997335000,7.033333e+07,3.331112e+08,1.000,3153335000,3.736,Adjust Tactics
9,Eon Productions,22,1046650000,4915989899,4.757500e+07,2.234541e+08,1.000,3869339899,3.697,Fund More


## 5. Results Summary

The highest-ROA studio in the dataset with a meaningful film count is Blumhouse Productions, whose
low-budget horror model (average budget of roughly $4.3M per film against a 95% probability of
profitability) produces a return on ad-spend-equivalent metric far above major studios that spend on
average 15 to 40 times more per film. This mirrors the source framework's core finding pattern: a
low-cost, high-conversion-probability publisher (here, a low-budget, high-hit-rate studio) can
outperform high-spend competitors on efficiency even without matching them on absolute volume.

In [6]:
genre_stats.to_csv("../data/genre_stats.csv", index=False)
studio_stats.to_csv("../data/studio_stats.csv", index=False)
print("Saved genre_stats.csv, studio_stats.csv")

Saved genre_stats.csv, studio_stats.csv
